In [11]:
import os
import json
from pydantic import BaseModel, ConfigDict
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

class Recipe(BaseModel):
    model_config = ConfigDict(extra="forbid")
    
    title: str
    ingredients: list[str]
    calories: int
    prep_time_minutes: int

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                { 
                    "role": "user",
                    "content": "Generate a healthy paneer sandwich recipe."
                }
            ],
            temperature=0.2,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "recipe",
                    "strict": True,
                    "schema": Recipe.model_json_schema()
                }
            }
        )

print(response.choices[0].message.content)

#convert  the JSON response into a Pydantic Recipe object using
recipe = Recipe.model_validate_json(
    response.choices[0].message.content
)

{"title":"Healthy Paneer Sandwich","ingredients":["Whole wheat bread slices","Paneer cubes","Cucumber slices","Tomato slices","Red onion slices","Fresh coriander leaves","Mint chutney","Low-fat yogurt","Olive oil spray","Salt","Black pepper","Paprika"],"calories":350,"prep_time_minutes":15}


In [12]:
recipe.calories

350